<a href="https://colab.research.google.com/github/Avni2000/cs639-final-project/blob/main/probe-baseline/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install pandas
%pip install transformers
%pip install torch
%pip install datasets
# restart the kernel after installing in your virtual environment to ensure the new packages are loaded

In [ ]:
import pandas as pd

## Grab our dataset of historical AIME problems

In [ ]:

from datasets import load_dataset

ds = load_dataset("gneubig/aime-1983-2024")

# get all features
filtered = ds['train']

# Save to csv
filtered.to_csv('aime_historical.csv')

df = pd.read_csv('./aime_historical.csv')
display(df.tail(50))

## Testing

Let's do a small test to see if our thinking traces works, we're calling HF correctly, etc.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"  # pull from HuggingFace
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, output_hidden_states=True)

inputs = tokenizer("Why is the sky blue?", return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

# outputs.hidden_states is a tuple: one tensor per layer
# each tensor shape: (batch, seq_len, hidden_dim)
hidden_states = outputs.hidden_states

for layer_idx, layer_hs in enumerate(hidden_states):
    print(f"Layer {layer_idx}: {layer_hs.shape}")
    # e.g. Layer 0: torch.Size([1, 8, 2048])
    #               batch ^ ^ tokens ^ ^ ^ hidden_dim

# store as list of tensors per token across all layers:
# shape: [num_layers][batch, token, hidden_dim]
all_layers = [hs.squeeze(0) for hs in hidden_states]  # drop batch dim
# all_layers[layer][token] → hidden vector for that token at that layer